# Collaboration Facilitator Agent

This notebook is used for the purpose of developing and validating the collaboration facilitator agent.

In [1]:
import json
from src import llms, agent, utils, models

## Mock LLM

In [2]:
# Load dataset from file
dataset = utils.load_dataset_from_jsonl('results/training_docs.jsonl')

if not dataset:
    print("Warning: results/training_docs.jsonl is empty. Please run the training_data_gen.ipynb notebook first.")
    exit()

print(f"Loaded {len(dataset)} documents.")

Loaded 100 documents.


In [3]:
def create_mock_responder(prompt: str, system_prompt=None, max_output_tokens: int = 800) -> str:
    """Custom responder for mock testing."""
    # Return intervention based on type requested
    if "You must use a compromise synthesis approach" in prompt:
        return json.dumps({
            "intervention": "How about this revised sentence: 'While leadership has mandated a 15% budget reduction, we will work with each team to manage the transition and mitigate any negative impact.'"
        })
    elif "You must use a Socratic questioning approach" in prompt:
        return json.dumps({
            "intervention": "It seems the core disagreement is between reflecting leadership's directive accurately and maintaining team morale. Could we explore alternative phrasing that does both?"
        })
    else:
        return json.dumps({
            "intervention": "NO_INTERVENTION"
        })

mock_llm = llms.MockLLMClient(responder=create_mock_responder)

print(agent.generate_single_intervention(mock_llm, dataset[0], 3, models.InterventionType.NO_INTERVENTION, True))


{"intervention": "NO_INTERVENTION"}


## End to End with OpenAi client

Requires OpenAI API key

In [ ]:
open_ai_llm = llms.OpenAiClient()

agent_intervention = agent.generate_single_intervention(llm=open_ai_llm, 
                                                        context=dataset[0],
                                                        comments_used=3,
                                                        intervention_type=models.InterventionType.SOCRATIC_QUESTIONING,
                                                        chain_of_thought=True)

print(agent_intervention)

{
  "reasoning": "The discussion shows a clear division between the author, who is focused on conveying the emotional response of the community, and the peer, who emphasizes the need for supporting evidence to strengthen the argument. Both viewpoints are valid, but there is ambiguity about how the emotional appeal can be balanced with factual support. A Socratic question could help both parties clarify their priorities and discover a mutual understanding of how best to present the opponents' concerns.",
  "intervention_type": "socratic",
  "comment": "What specific types of evidence or examples do you think would effectively illustrate the opponents' concerns while still maintaining the emotional impact of their viewpoint?"
}
